# From Weak Signals to Decisions: Google Trends for Product Thinking

Search behavior is not a perfect proxy for demand, but it is one of the fastest public signals available to product teams.

This notebook uses weekly Google Trends snapshots from France to compare three kinds of attention:

- **ChatGPT**: fast-moving technology adoption.
- **iPhone**: product-cycle and launch-driven attention.
- **Weather / météo**: stable, seasonal, recurring intent.

The goal is not to forecast the future with false precision. The goal is to turn weak public signals into better product questions, sharper timing decisions, and more honest uncertainty.

## Kaggle publication angle

This notebook is useful because it shows a compact, reusable workflow for public signal analysis:

1. Load a small, documented dataset.
2. Compare short-term and long-term attention patterns.
3. Separate recurring seasonality from unusual shifts.
4. Translate charts into product decisions.
5. Document limitations clearly.

It is designed for students, product builders, analysts, and Kaggle users who want practical notebooks that are simple enough to reuse but strong enough to inform real decisions.

## Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30)

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.titleweight"] = "bold"

## Load the dataset

The notebook runs on Kaggle and locally. On Kaggle, it searches under `/kaggle/input`. Locally, it falls back to the repository package or the original snapshots.

In [ ]:
def find_dataset_file(pattern: str) -> Path:
    search_roots = [
        Path("/kaggle/input"),
        Path.cwd() / "kaggle" / "google_trends_dataset",
        Path.cwd().parent / "kaggle" / "google_trends_dataset",
        Path.cwd() / "data" / "snapshots",
        Path.cwd().parent / "data" / "snapshots",
    ]
    for root in search_roots:
        if root.exists():
            matches = sorted(root.rglob(pattern))
            if matches:
                return matches[0]
    raise FileNotFoundError(f"Could not find {pattern}. Add the Kaggle dataset or run from the repo root.")

path_5y = find_dataset_file("iot_FR_today_5-y_chatgpt_iphone_meteo.csv")
path_12m = find_dataset_file("iot_FR_today_12-m_chatgpt_iphone_meteo.csv")

path_5y, path_12m

In [ ]:
df_5y_raw = pd.read_csv(path_5y)
df_12m_raw = pd.read_csv(path_12m)

print("5-year snapshot:", df_5y_raw.shape)
print("12-month snapshot:", df_12m_raw.shape)
df_5y_raw.head()

## Cleaning and preparation

The source is already clean, so preparation should stay boring: parse dates, validate ranges, reshape to long format, and create rolling averages. Boring preparation is a feature, not a weakness.

In [ ]:
SIGNALS = ["chatgpt", "iphone", "meteo"]


def prepare_trends(raw: pd.DataFrame, window_label: str) -> pd.DataFrame:
    df = raw.copy()
    df.columns = df.columns.str.strip().str.lower()
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    for col in SIGNALS:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)
    df[SIGNALS] = df[SIGNALS].ffill().bfill()

    assert df["date"].is_monotonic_increasing
    assert df[SIGNALS].notna().all().all()
    assert df[SIGNALS].min().min() >= 0
    assert df[SIGNALS].max().max() <= 100

    long = df.melt(id_vars="date", value_vars=SIGNALS, var_name="signal", value_name="interest")
    long["window"] = window_label
    long["rolling_4w"] = long.groupby("signal")["interest"].transform(lambda s: s.rolling(4, min_periods=1).mean())
    long["rolling_13w"] = long.groupby("signal")["interest"].transform(lambda s: s.rolling(13, min_periods=1).mean())
    return long

trends_5y = prepare_trends(df_5y_raw, "5 years")
trends_12m = prepare_trends(df_12m_raw, "12 months")
trends_5y.head()

In [ ]:
quality = trends_5y.groupby("signal").agg(
    observations=("interest", "size"),
    first_date=("date", "min"),
    last_date=("date", "max"),
    min_interest=("interest", "min"),
    median_interest=("interest", "median"),
    max_interest=("interest", "max"),
).reset_index()
quality

## Exploratory analysis: three different attention patterns

A useful weak-signal analysis starts by comparing shapes, not just rankings. The same score can mean different things depending on whether the signal is seasonal, launch-driven, or structurally emerging.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
sns.lineplot(data=trends_5y, x="date", y="rolling_13w", hue="signal", ax=ax)
ax.set_title("Long-term attention patterns in France")
ax.set_xlabel("")
ax.set_ylabel("13-week rolling Google Trends index")
ax.legend(title="Signal")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
sns.lineplot(data=trends_12m, x="date", y="rolling_4w", hue="signal", ax=ax)
ax.set_title("Recent attention: last 12 months")
ax.set_xlabel("")
ax.set_ylabel("4-week rolling Google Trends index")
ax.legend(title="Signal")
plt.show()

## Comparing ChatGPT vs iPhone vs Weather

For product thinking, the comparison is the point:

- **Weather** is recurring intent: high, stable, seasonal.
- **iPhone** is product-cycle attention: bursts around launches and news.
- **ChatGPT** is adoption attention: newer, more volatile, and useful for detecting changing expectations around AI.

In [ ]:
summary = trends_5y.groupby("signal").agg(
    average_interest=("interest", "mean"),
    median_interest=("interest", "median"),
    volatility=("interest", "std"),
    peak_interest=("interest", "max"),
    latest_interest=("interest", lambda s: s.iloc[-1]),
).round(2)
summary["latest_vs_median"] = (summary["latest_interest"] - summary["median_interest"]).round(2)
summary.sort_values("average_interest", ascending=False)

In [ ]:
peak_table = (
    trends_5y.loc[trends_5y.groupby("signal")["interest"].idxmax(), ["signal", "date", "interest"]]
    .sort_values("interest", ascending=False)
    .reset_index(drop=True)
)
peak_table

## Seasonality and attention shifts

A weak signal becomes more useful when we ask whether the current level is normal for the signal. A weather spike may be expected. A ChatGPT spike may indicate a new product behavior, media cycle, or adoption question worth investigating.

In [ ]:
weekly = trends_5y.copy()
weekly["week_of_year"] = weekly["date"].dt.isocalendar().week.astype(int)
seasonality = weekly.groupby(["signal", "week_of_year"], as_index=False)["interest"].median()

fig, ax = plt.subplots(figsize=(13, 5))
sns.lineplot(data=seasonality, x="week_of_year", y="interest", hue="signal", ax=ax)
ax.set_title("Typical seasonal profile by week of year")
ax.set_xlabel("Week of year")
ax.set_ylabel("Median Google Trends index")
ax.legend(title="Signal")
plt.show()

In [ ]:
baseline = trends_5y.groupby("signal")["interest"].median()
latest = trends_5y.sort_values("date").groupby("signal").tail(1).set_index("signal")["interest"]
recent = trends_5y.sort_values("date").groupby("signal").tail(8).groupby("signal")["interest"].mean()

shift_score = pd.DataFrame({
    "median_baseline": baseline,
    "latest": latest,
    "recent_8w_average": recent,
})
shift_score["latest_minus_median"] = shift_score["latest"] - shift_score["median_baseline"]
shift_score["recent_minus_median"] = shift_score["recent_8w_average"] - shift_score["median_baseline"]
shift_score.sort_values("recent_minus_median", ascending=False).round(2)

## Weak signal interpretation

This dataset supports a practical reading:

- **Weather** gives a benchmark for recurring, high-intent behavior. It is not surprising when it dominates attention.
- **iPhone** helps illustrate event-driven attention. Peaks are often useful for campaign timing and launch analysis.
- **ChatGPT** is the strategic signal. It may not always dominate, but changes in its baseline can reveal changing expectations around AI in everyday tools.

The best product question is not “which line is highest?” It is “which change should alter our next decision?”

In [ ]:
def classify_signal(row):
    if row["volatility"] >= 20 and row["latest_vs_median"] > 0:
        return "active shift"
    if row["volatility"] >= 20:
        return "event-driven"
    if row["median_interest"] >= 30:
        return "recurring intent"
    return "low or emerging attention"

interpretation = summary.copy()
interpretation["product_reading"] = interpretation.apply(classify_signal, axis=1)
interpretation.sort_values(["product_reading", "average_interest"])

## Decision framing

A product team should turn weak signals into decisions with explicit uncertainty.

| Signal pattern | Product question | Example decision |
|---|---|---|
| Recurring high intent | Are we present when users repeatedly need help? | Plan seasonal content, SEO, support, alerts. |
| Event-driven peaks | Can we prepare around known launch moments? | Time campaigns, landing pages, comparisons. |
| Emerging or shifting baseline | Is user behavior structurally changing? | Prototype features, run discovery, monitor adoption. |

## Practical product recommendation

If this were a product strategy review, the recommendation would be:

1. Treat **weather** as the control signal: recurring public intent with strong seasonal behavior.
2. Treat **iPhone** as the launch/event signal: useful for timing and campaign readiness.
3. Treat **ChatGPT** as the strategic weak signal: monitor baseline changes, not only peaks.
4. Build a lightweight monthly signal review: trend chart, peak table, shift score, decision note.
5. Do not make product bets from Trends alone. Use it to prioritize research, not to replace research.

## Limitations of Google Trends

Google Trends is powerful, but easy to overread.

- The values are relative indexes, not absolute search counts.
- The scale depends on query set, geography, and timeframe.
- Search interest is not the same as demand, revenue, satisfaction, or adoption.
- A peak can be caused by news, controversy, curiosity, or genuine intent.
- Ambiguous terms need domain knowledge before interpretation.

The responsible use is directional: detect questions worth asking, then validate with stronger evidence.

## Conclusion

Weak signals are valuable when they improve decisions, not when they create dramatic charts.

This notebook shows a reusable product-thinking workflow:

1. Compare signals over multiple time windows.
2. Separate stable seasonality from unusual attention shifts.
3. Convert observations into product questions.
4. State a recommendation with limits.

For Kaggle, this is also a compact portfolio asset: a documented dataset, a clean public notebook, and a practical analytical story that others can reuse.